# SQL mini cheatsheet

# Содержание

## 1. CREATE TABLE. 

### 1.1. Создание таблицы, добавление данных, работа  с таблицей

### 1.2. Константы 

### 1.3. Функции одного аргумента

## 2. SELECT 

### 2.1. В общем виде

### 2.2. Оформляющие конструкции (AS, CASE, distinct)

### 2.3. group by и агрегирующие функции. Where vs Having

### 2.4. Подзапросы

### 2.5. with

## 3. JOIN

### 3.1. Биективное соединение по ключу

### 3.2. left/right/outer/inner join

### 3.3. Конкатенация результатов

## 4. ОКОННЫЕ ФУНКЦИИ

### 4.1. Общая конструкция

### 4.2. function

### 4.3. ROWS/RANGE BETWEEN

# 1. CREATE TABLE. Константы. Функции одного аргумента

**1.1. создание таблицы**

CREATE TABLE customers (

    customer_id   INT PRIMARY KEY,
    
    customer_name  VARCHAR(100) NOT NULL,
        
    signup_date    DATE NOT NULL

);

типы переменных INT, VARCHAR(100), DATE,...

ограничения PRIMARY/FOREIGN KEY, UNIQUE...

**добавление**

    INSERT INTO customers (customer_name, customer_id) VALUES ('Аня', 500), 
    ('Лена', 1500), 
    ('Вася', 25000);

**удаление**

    DELETE FROM customers;

    DELETE FROM customers WHERE customer_id = 500;

**изменение**

    UPDATE products
    SET price = price * 1.1
    FROM categories
    WHERE categories.id = products.category_id AND categories.name = 'Электроника';

    ALTER TABLE customers ADD COLUMN phone VARCHAR(20);
    ALTER TABLE customers DROP COLUMN phone;


**1.2. Константы**

    TRUE, FALSE, CURRENT_DATE, CURRENT_TIMESTAMP, 

**NULL**

NULL нельзя проверять через = NULL или <> NULL; используются IS NULL и IS NOT NULL

COALESCE(value, replacement) — замена NULL;

NULLIF(a, b) — превращает a в NULL, если a = b;

**1.3. Функции одного аргумента**

**Функции чисел**

    ROUND(3.1415, 2) 

    FLOOR(5.8)

**Функции строк**

    RIGHT(column_name, 3) = 'abc';

    name LIKE '%Salzburg%'

    LENGTH('abc')


# 2. SELECT 

**2.1. В общем виде.**

    SELECT ...
    FROM ... / JOIN ...
    WHERE ...
    GROUP BY ... 
    HAVING ...
    ORDER BY ... DESC/ASC
    LIMIT ...;

**выводит все**

    SELECT * from...;  

**2.2. оформляющие конструкции (AS, CASE, distinct)**


    SELECT DISTINCT
        product_name,
        price,
        price * 1.20 AS price_with_tax,
        CASE
            WHEN price < 1000 THEN 'budget'
            WHEN price < 5000 THEN 'standard'
            ELSE 'premium'
        END AS price_segment
    FROM products;
    
AS — переименование столбца или таблицы;

CASE — условная логика; CASE WHEN X1 THEN Y1 WHEN X2 THEN Y2... ELSE YN END

DISTINCT — удаление дублей в проекции (но не исправление ошибок JOIN, см. далее)

**2.3. group by и агрегирующие функции**

COUNT() — считает число строк в группе. SUM() — считает сумму чисел в столбце. AVG() — считает среднее значение. MIN() / MAX() — очевидно.

! игнорируют NULL

! where используется для изначальных значений, having - для агрегирующих функций

    SELECT 
        betting_type,
        ROUND(AVG(bet_amount), 2) AS avg_amount
    FROM 
        bets
    GROUP BY 
        betting_type
    ORDER BY 
        avg_amount ASC;
    
    SELECT user_id, count(transaction_id) AS transactions_count 
    FROM transactions WHERE transaction_date >= '2024-01-01' 
      AND transaction_date < '2024-02-01'
      GROUP BY user_id
      HAVING count(transaction_id) > 5
      Order by user_id ASC


**2.4. подзапросы**

    SELECT name FROM employees
    WHERE salary > (SELECT AVG(salary) FROM employees);

    -- Подзапрос в FROM (как таблица)
    SELECT * FROM (
        SELECT user_id, COUNT(*) as order_count
        FROM orders
        GROUP BY user_id
    ) AS user_orders
    WHERE order_count > 5;

**2.5. With**

    WITH ranked_orders AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY order_date DESC) as rn
        FROM orders
    )
    SELECT * FROM ranked_orders WHERE rn = 1;


# 3. JOIN

**3.1. биективное соединение по ключу**

Пример 1.

    SELECT Countries.name AS country_name
    FROM Cities 
    JOIN Regions  ON Cities.regionid=Regions.id
    JOIN Countries  ON Countries.id=Regions.countryid
    WHERE Cities.name = 'Salzburg';

Пример 2.

    SELECT e.name,
    COALESCE(SUM(o.amount), 0) AS total_amount 
    FROM employees e
    LEFT JOIN (
    select * from orders WHERE created_at >= '2024-01-01'
    AND created_at < '2025-01-01') as o 
    ON  e.id = o.employee_id
    GROUP BY e.id, e.name

Пример 3.

    SELECT
        o.order_id,
        o.order_date,
        c.customer_name,
        p.product_name,
        oi.quantity,
        oi.quantity * oi.unit_price AS line_amount
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    JOIN order_items oi ON oi.order_id = o.order_id
    JOIN products p ON p.product_id = oi.product_id;

**3.2. left/right/outer/inner/cross join**

Визуализация всех типов соединения

https://github.com/fufaevvlvl/CheatSheets/blob/main/A_Join_Visualization.ipynb

**3.3. Конкатенация результатов**

-- UNION (объединение без дублей)

    SELECT name FROM employees
    UNION
    SELECT name FROM managers;

-- UNION ALL (объединение с дублями)

    SELECT name FROM employees
    UNION ALL
    SELECT name FROM managers;

-- INTERSECT (пересечение)

    SELECT name FROM employees
    INTERSECT
    SELECT name FROM managers;

-- EXCEPT (разность)

    SELECT name FROM employees
    EXCEPT
    SELECT name FROM managers;

# 4. ОКОННЫЕ ФУНКЦИИ

**4.1. Общая конструкция** 

Ключевое слово OVER() является обязательным маркером оконной конструкции и отделяет вычисления от обычных скалярных операций, при этом пустые скобки означают окно из всех строк результирующего набора.

function(...) OVER ( 

    PARTITION BY ...
    ORDER BY ...     
    ROWS BETWEEN ...
)

Оконная функция в SQL состоит из пяти ключевых элементов, каждый из которых строго определяет её поведение.

1. function - Сама функция (например, SUM, ROW_NUMBER, LAG) задаёт тип вычисления — агрегатное, ранжирующее или смещения, и применяется не к группе в целом, а к динамическому набору строк, связанному с текущей записью.

2. Предложение PARTITION BY разбивает строки на независимые логические группы (партиции), внутри которых функция перезапускается, — аналогично GROUP BY, но без свёртки, так что каждая строка сохраняется, а расчёт ведётся в пределах своей партиции.

4. Предложение ORDER BY внутри OVER устанавливает порядок строк внутри партиции (или всего набора), что критически важно для ранжирующих функций, а для агрегатов включает режим накопительного итога по умолчанию.

5. Опциональная рамка (ROWS | RANGE BETWEEN ...), задаваемая после ORDER BY, явно ограничивает множество соседних строк, участвующих в вычислении для текущей строки, позволяя реализовать скользящие средние, кумулятивные суммы с границами или сравнения с фиксированным смещением.

нумеруем сотрудников по убыванию зарплаты, причём отдельно для каждого отдела:

    SELECT
    name,    
    department,
    salary,    
    RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dept_rank
    FROM employees;

Накопительный итог

    SELECT 
    date,    
    sales,    
    SUM(sales) OVER (ORDER BY date) AS total,    
    AVG(sales) OVER (ORDER BY date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS 3_day_av
    FROM daily_sales    

**4.2. function **

    · Ранжирующие: ROW_NUMBER(), RANK(), DENSE_RANK().
    · Функции смещения: LAG(), FIRST_VALUE(), LAST_VALUE().
    · Агрегатные: SUM(), AVG(), COUNT().

**без аргументов**

ROW_NUMBER() Присваивает уникальный порядковый номер каждой строке. “Ничьи” разрываются произвольно (или по вторичному ORDER BY). 1, 2, 3, 4 Удаление дубликатов, выбор ровно одной строки на группу.
RANK() Присваивает одинаковый ранг дубликатам, но пропускает следующие номера (создает “дыры” в нумерации). 1, 2, 2, 4 Спортивные рейтинги (два серебряных призера, следующего бронзового нет).
DENSE_RANK() Присваивает одинаковый ранг дубликатам, но не пропускает номера (нумерация идет плотно). 1, 2, 2, 3 Поиск топ-N элементов в каждой категории без разрывов в рангах.

    WITH employees AS (
        SELECT 'Оля' AS name, 'IT' AS dept, 7000 AS salary UNION ALL
        SELECT 'Вася', 'IT', 6000 UNION ALL
        SELECT 'Петя', 'IT', 6000 UNION ALL  -- Зарплата такая же, как у Васи
        SELECT 'Аня', 'IT', 5000
    )
    SELECT 
        name,
        dept,
        salary,
        ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) AS row_num,
        RANK()       OVER (PARTITION BY dept ORDER BY salary DESC) AS rnk,
        DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS dense_rnk
    FROM employees;
    
**LAG**    

    SELECT 
        month,
        revenue,
        LAG(revenue, 1) OVER (ORDER BY month) AS prev_month_revenue,
        ROUND((revenue - LAG(revenue, 1) OVER (ORDER BY month)) * 100.0 / LAG(revenue, 1) OVER (ORDER BY month), 2) AS growth_pct
    FROM monthly_metrics;

**4.3. ROWS/RANGE BETWEEN**

    UNBOUNDED PRECEDING Самая первая строка в текущей партиции (начало окна).
    N PRECEDING N строк физически перед текущей строкой (например, 2 PRECEDING).
    CURRENT ROW Текущая строка, для которой производится вычисление.
    N FOLLOWING N строк физически после текущей строки (например, 1 FOLLOWING).
    UNBOUNDED FOLLOWING Самая последняя строка в текущей партиции (конец окна).

Главное отличие:  ROWS  vs  RANGE 
 •  ROWS  работает с физическими строками. Он считает строго количество строк (смещение), игнорируя их значения.
 •  RANGE  работает с логическими значениями. Он включает в рамку все строки, которые имеют такое же значение в колонке  ORDER BY , как и текущая строка (группирует “ничьи”).

    SELECT 
    date,
    daily_revenue,
    -- Среднее за 3 дня: вчера, сегодня и завтра (или 2 дня до + текущий)
    ROUND(AVG(daily_revenue) OVER (
        ORDER BY date 
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS moving_avg_3_days
    FROM sales_data
    ORDER BY date;